# Capstone Project DB15-G009: Toko Firaza — AI Serving & Pipeline (Google Colab)

Notebook ini dirancang khusus untuk dijalankan di **Google Colab (GPU T4 / L4)** (menggantikan Kaggle Kernel agar pengaturan port dan tunnel lebih leluasa).

Arsitektur sesuai arahan Advisor:
1. **Serving LLM Langsung dengan vLLM + Ngrok (Tanpa 9router):** Menjalankan OpenAI-compatible API server di Colab GPU.
2. **Structured Output Kontrak Backend:** Menggunakan **Instructor + Pydantic** (bukan *few-shot prompting* manual). Kategori pesanan langsung ditentukan di dalam `StructuredOrderSummary`.
3. **RAG Framework & Vector Database:** Menggunakan **LlamaIndex + ChromaDB** (menggantikan FAISS).
4. **Observability & Tracing:** Terintegrasi **Langfuse** untuk memantau latensi, token, dan validasi kontrak.

In [ ]:
# 1. Cek GPU Colab & Install Dependencies (vLLM, Ngrok, Instructor, LlamaIndex, ChromaDB, Langfuse)
!nvidia-smi
!pip install -q vllm pyngrok instructor openai pydantic chromadb llama-index-core llama-index-vector-stores-chroma llama-index-llms-openai-like llama-index-embeddings-huggingface langfuse scikit-learn

In [ ]:
# 2. Jalankan vLLM OpenAI-Compatible Server di Background & Ekspos Langsung via Ngrok (Tanpa 9router)
import os, subprocess, time, urllib.request
from pyngrok import ngrok

# Masukkan token Ngrok Anda (dapatkan di https://dashboard.ngrok.com)
NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN", "ISI_TOKEN_NGROK_ANDA")
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
VLLM_PORT = 8000

if NGROK_AUTHTOKEN and NGROK_AUTHTOKEN != "ISI_TOKEN_NGROK_ANDA":
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

vllm_proc = subprocess.Popen([
    "vllm", "serve", MODEL_ID,
    "--host", "0.0.0.0",
    "--port", str(VLLM_PORT),
    "--dtype", "half",
    "--max-model-len", "4096",
    "--api-key", "token-firaza-vllm",
])

print("Menunggu vLLM server siap...")
for _ in range(60):
    try:
        req = urllib.request.Request(
            f"http://127.0.0.1:{VLLM_PORT}/v1/models",
            headers={"Authorization": "Bearer token-firaza-vllm"}
        )
        with urllib.request.urlopen(req, timeout=2) as r:
            if r.status == 200:
                print("[OK] vLLM Server aktif!")
                break
    except Exception:
        time.sleep(5)

public_tunnel = ngrok.connect(VLLM_PORT, "http")
print("\n=== SALIN KONFIGURASI INI KE FILE .env BACKEND LOKAL ===")
print(f"LLM_BASE_URL={public_tunnel.public_url}/v1")
print("LLM_API_KEY=token-firaza-vllm")
print(f"LLM_MODEL={MODEL_ID}")

In [ ]:
# 3. Uji Coba Kontrak Pydantic + Instructor + Langfuse terhadap vLLM Server
import json
import instructor
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Literal, Optional

OrderCategory = Literal["Ready Stock", "Custom Sajadah", "Custom Handuk", "Hampers", "Other"]

class OrderItem(BaseModel):
    product_type: str
    quantity: int = Field(default=1, ge=1)
    color: Optional[str] = None
    size: Optional[str] = None
    custom_text: Optional[str] = None
    font_style: Optional[str] = None

class OrderSummary(BaseModel):
    order_category: OrderCategory
    customer_name: Optional[str] = None
    items: List[OrderItem]

client = instructor.from_openai(
    OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="token-firaza-vllm"),
    mode=instructor.Mode.JSON,
)

sample_chat = (
    "Assalamualaikum kak, saya Farhan mau pesen hampers lebaran 3 paket wrn hijau emerald. "
    "Isinya sajadah custom uk dewasa bdr nama 'Keluarga Sutrisno' font kaligrafi arab."
)

result = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct",
    response_model=OrderSummary,
    max_retries=2,
    messages=[{"role": "user", "content": sample_chat}],
)
print(json.dumps(result.model_dump(), indent=2, ensure_ascii=False))